# Final Model Evaluation on Test Set

Evaluate the best models from each approach:
1. Naive Baseline
2. Best LSTM Model
3. Best Chronos2 Model (Zero-Shot and Fine-Tuned)

**Goal**: Determine the best performing model on the held-out test set.

In [ ]:
import os
import sys
import json
import glob
import pandas as pd
import numpy as np
import torch
from tqdm.auto import tqdm
from chronos import Chronos2Pipeline
from darts.models import BlockRNNModel
from darts import TimeSeries
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

if os.path.basename(os.getcwd()) == 'notebooks':
    project_root = os.path.abspath('..')
else:
    project_root = os.getcwd()

if project_root not in sys.path:
    sys.path.append(project_root)

from src.datamodule import ElectricityDataModule

# Define simple sMAPE function (without masking)
def simple_smape(y_true: torch.Tensor, y_pred: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    """
    Calculates the Symmetric Mean Absolute Percentage Error (sMAPE).
    
    Args:
        y_true (torch.Tensor): The ground truth values. Shape: (B, T, C).
        y_pred (torch.Tensor): The predicted values. Shape: (B, T, C).
        eps (float): A small epsilon value to avoid division by zero.
    
    Returns:
        torch.Tensor: A scalar tensor representing the mean sMAPE.
    """
    num = torch.abs(y_true - y_pred)
    den = torch.abs(y_true) + torch.abs(y_pred) + eps
    smape = 2.0 * num / den
    return smape.mean()

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Configuration

In [ ]:
BASE_DIR = ".."
DATA_DIR = os.path.join(BASE_DIR, "data")
TEST_DIR = os.path.join(DATA_DIR, "test")
SCALERS_DIR = os.path.join(DATA_DIR, "scalers")
MODELS_DIR = os.path.join(BASE_DIR, "models")
RESULTS_DIR = os.path.join(BASE_DIR, "results")

TARGET_COLS = ["high", "low", "close", "volume"]
INPUT_CHUNK_LENGTH = 48
OUTPUT_CHUNK_LENGTH = 10
SEED = 827
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Preprocess test data to add is_trading column if missing
print("Checking test data for is_trading column...")
test_files = glob.glob(os.path.join(TEST_DIR, "*.parquet"))
need_preprocessing = False

# Check if any file is missing is_trading
for test_file in test_files[:3]:  # Check first 3 files
    df = pd.read_parquet(test_file)
    if 'is_trading' not in df.columns:
        need_preprocessing = True
        break

if need_preprocessing:
    print("Adding is_trading column to test data...")
    for test_file in tqdm(test_files, desc="Processing test files"):
        df = pd.read_parquet(test_file)
        if 'is_trading' not in df.columns:
            # is_trading = 1 if any target column has non-zero value
            is_trading = (df[TARGET_COLS].abs().sum(axis=1) > 0).astype(int)
            df['is_trading'] = is_trading
            df.to_parquet(test_file, index=False)
    print("✓ Added is_trading column to all test files")
else:
    print("✓ Test data already has is_trading column")

# Load baseline results to get best model info
with open(os.path.join(RESULTS_DIR, "baseline_summary.json"), "r") as f:
    baseline_summary = json.load(f)

# Load Chronos2 results
with open(os.path.join(RESULTS_DIR, "chronos2_results.json"), "r") as f:
    chronos2_summary = json.load(f)

print("\nConfiguration loaded successfully")
print(f"Test data directory: {TEST_DIR}")
print(f"Device: {device}")

In [ ]:
# Initialize ElectricityDataModule for test data
datamodule = ElectricityDataModule(
    train_parquet=os.path.join(DATA_DIR, "train_trading_only"),  # Not used but required
    val_parquet=os.path.join(DATA_DIR, "val_trading_only"),      # Not used but required
    test_parquet=TEST_DIR,
    scalers_dir=SCALERS_DIR,
    batch_size=32,  # Use smaller batches for test evaluation
    num_workers=4,
    dataset_kwargs={
        'input_chunk_length': INPUT_CHUNK_LENGTH,
        'output_chunk_length': OUTPUT_CHUNK_LENGTH,
        'target_cols': TARGET_COLS,
        'stride': OUTPUT_CHUNK_LENGTH,  # Non-overlapping windows for clean evaluation
        'shuffle_buffer': 0  # No shuffling for test set
    }
)

# Setup the test dataset
datamodule.setup(stage='test')

# Get the test dataloader
test_dataloader = datamodule.test_dataloader()

print(f"Test dataloader ready")
print(f"Batch size: {datamodule.batch_size}")
print(f"Input chunk length: {INPUT_CHUNK_LENGTH}")
print(f"Output chunk length: {OUTPUT_CHUNK_LENGTH}")

## 2. Setup Test Data

In [ ]:
print("="*80)
print("EVALUATING AUTOGLUON MODEL")
print("="*80)

# Path to AutoGluon model
AUTOGLUON_DIR = os.path.join(MODELS_DIR, "autogluon_quick_test")

# Check if AutoGluon model exists
if os.path.exists(AUTOGLUON_DIR):
    try:
        print(f"Loading AutoGluon predictor from: {AUTOGLUON_DIR}")
        
        # Load the predictor
        autogluon_predictor = TimeSeriesPredictor.load(AUTOGLUON_DIR)
        print(f"✓ Loaded AutoGluon predictor")
        print(f"  Prediction length: {autogluon_predictor.prediction_length}")
        print(f"  Best model: {autogluon_predictor.model_best}")
        
        # Convert test data to AutoGluon format
        # We need to load test parquet files and convert them
        print("\nConverting test data to AutoGluon format...")
        test_files = sorted(glob.glob(os.path.join(TEST_DIR, "*.parquet")))
        
        # Load a subset for testing (can be adjusted)
        all_test_data = []
        exclude_cols = TARGET_COLS + ['ExecutionTime']
        
        for test_file in tqdm(test_files, desc="Loading test files"):
            asset_name = os.path.basename(test_file).replace('.parquet', '')
            df = pd.read_parquet(test_file)
            
            # Take last 500 rows for faster evaluation
            df = df.tail(500).copy()
            
            # Clean timestamps
            df['ExecutionTime'] = pd.to_datetime(df['ExecutionTime'])
            if df['ExecutionTime'].dt.tz is not None:
                df['ExecutionTime'] = df['ExecutionTime'].dt.tz_localize(None)
            
            # Get covariate columns
            covariate_cols = [c for c in df.columns if c not in exclude_cols]
            
            # Convert to float32 for consistency
            if 'is_trading' in df.columns:
                df['is_trading'] = df['is_trading'].astype(np.float32)
            
            # Create separate entries for each target
            for col in TARGET_COLS:
                if col in df.columns:
                    item_df = pd.DataFrame({
                        'timestamp': df['ExecutionTime'].values,
                        'item_id': f"{asset_name}_{col}",
                        'target': df[col].astype(np.float32).values
                    })
                    
                    # Add covariates
                    for cov_col in covariate_cols:
                        if cov_col in df.columns:
                            item_df[cov_col] = df[cov_col].values
                    
                    # Drop NaN rows
                    item_df = item_df.replace([np.inf, -np.inf], np.nan).dropna()
                    all_test_data.append(item_df)
        
        # Combine into single DataFrame
        test_combined_df = pd.concat(all_test_data, ignore_index=True)
        print(f"Combined test data shape: {test_combined_df.shape}")
        
        # Convert to TimeSeriesDataFrame
        test_ts_df = TimeSeriesDataFrame.from_data_frame(
            test_combined_df,
            id_column='item_id',
            timestamp_column='timestamp'
        )
        
        print(f"✓ Created TimeSeriesDataFrame with {len(test_ts_df.item_ids)} items")
        
        # Make predictions
        print("\nGenerating AutoGluon predictions...")
        autogluon_predictions = autogluon_predictor.predict(test_ts_df)
        print(f"✓ Generated predictions: {autogluon_predictions.shape}")
        
        # Convert predictions back to our format for sMAPE calculation
        # Organize by target variable
        autogluon_forecasts_by_target = {target: [] for target in TARGET_COLS}
        autogluon_ground_truths_by_target = {target: [] for target in TARGET_COLS}
        
        # Extract predictions for each item_id
        for item_id in autogluon_predictions.item_ids:
            # Parse asset and target from item_id (format: assetname_targetcol)
            parts = item_id.rsplit('_', 1)
            if len(parts) == 2:
                target_col = parts[1]
                if target_col in TARGET_COLS:
                    # Get predictions for this item
                    item_preds = autogluon_predictions.loc[item_id]['mean'].values
                    
                    # Get ground truth from test_ts_df
                    item_test_data = test_ts_df.loc[item_id]
                    # Get the last OUTPUT_CHUNK_LENGTH values as ground truth
                    item_gt = item_test_data['target'].values[-OUTPUT_CHUNK_LENGTH:]
                    
                    if len(item_preds) == OUTPUT_CHUNK_LENGTH and len(item_gt) == OUTPUT_CHUNK_LENGTH:
                        autogluon_forecasts_by_target[target_col].append(item_preds)
                        autogluon_ground_truths_by_target[target_col].append(item_gt)
        
        # Calculate sMAPE for AutoGluon
        autogluon_by_target = {}
        all_autogluon_forecasts = []
        all_autogluon_gts = []
        
        for target_name in TARGET_COLS:
            if autogluon_forecasts_by_target[target_name]:
                forecasts = np.array(autogluon_forecasts_by_target[target_name])
                gts = np.array(autogluon_ground_truths_by_target[target_name])
                
                # Convert to tensors
                forecast_t = torch.from_numpy(forecasts).float().to(device).unsqueeze(-1)
                gt_t = torch.from_numpy(gts).float().to(device).unsqueeze(-1)
                
                # Calculate sMAPE
                target_smape = simple_smape(gt_t, forecast_t).item() * 100
                autogluon_by_target[target_name] = target_smape
                
                all_autogluon_forecasts.append(forecast_t)
                all_autogluon_gts.append(gt_t)
        
        # Overall sMAPE
        if all_autogluon_forecasts:
            combined_forecasts = torch.cat(all_autogluon_forecasts, dim=-1)
            combined_gts = torch.cat(all_autogluon_gts, dim=-1)
            autogluon_smape = simple_smape(combined_gts, combined_forecasts).item() * 100
        else:
            autogluon_smape = float('inf')
        
        print(f"\nAutoGluon Model sMAPE: {autogluon_smape:.2f}%")
        print("\nPer-target sMAPE:")
        for target_name, smape in autogluon_by_target.items():
            print(f"  {target_name:8s}: {smape:.2f}%")
        
    except Exception as e:
        print(f"✗ Error evaluating AutoGluon model: {e}")
        import traceback
        traceback.print_exc()
        autogluon_smape = float('inf')
        autogluon_by_target = {}
else:
    print(f"✗ AutoGluon model not found at: {AUTOGLUON_DIR}")
    print("   Run the 05_Small_AutoGluon_Test.ipynb notebook first to train the model")
    autogluon_smape = float('inf')
    autogluon_by_target = {}

print("="*80)

## 3. Evaluate Naive Baseline

## 4. Evaluate AutoGluon Model

In [ ]:
print("="*80)
print("EVALUATING NAIVE BASELINE")
print("="*80)

def naive_forecast_batch(context_batch):
    """
    Naive forecasting: repeat last observed values.
    
    Args:
        context_batch: Tensor of shape [batch_size, context_len, n_features]
    
    Returns:
        forecast: Tensor of shape [batch_size, pred_len, n_features]
    """
    # Get last values along the time dimension
    last_values = context_batch[:, -1:, :]  # [batch_size, 1, n_features]
    # Repeat for prediction length
    forecast = last_values.repeat(1, OUTPUT_CHUNK_LENGTH, 1)  # [batch_size, pred_len, n_features]
    return forecast


# Collect all predictions and ground truths from the dataloader
all_naive_forecasts = []
all_ground_truths = []

print("Generating naive forecasts from test dataloader...")
for batch in tqdm(test_dataloader, desc="Naive forecasting"):
    past, past_mask, future, future_mask, asset_ids, past_ts_list = batch
    
    # Generate naive forecast (repeat last value)
    naive_pred = naive_forecast_batch(past)
    
    # Store predictions and ground truths
    all_naive_forecasts.append(naive_pred)
    all_ground_truths.append(future)

# Concatenate all batches
naive_forecasts = torch.cat(all_naive_forecasts, dim=0).to(device)
naive_ground_truths = torch.cat(all_ground_truths, dim=0).to(device)

print(f"\nCollected {naive_forecasts.shape[0]} test samples")
print(f"Shape: {naive_forecasts.shape}")

# Compute overall sMAPE (without masking)
naive_smape = simple_smape(naive_ground_truths, naive_forecasts).item() * 100

# Compute per-target sMAPE
naive_by_target = {}
for i, target_name in enumerate(TARGET_COLS):
    target_forecast = naive_forecasts[:, :, i:i+1]
    target_gt = naive_ground_truths[:, :, i:i+1]
    target_smape = simple_smape(target_gt, target_forecast).item() * 100
    naive_by_target[target_name] = target_smape

print(f"\nNaive Baseline sMAPE: {naive_smape:.2f}%")
print("\nPer-target sMAPE:")
for target_name, smape in naive_by_target.items():
    print(f"  {target_name:8s}: {smape:.2f}%")
print("="*80)

# Store these for use in other evaluations
naive_ground_truths_t = naive_ground_truths
naive_forecasts_t = naive_forecasts

## 5. Evaluate Best LSTM Model

In [ ]:
print("="*80)
print("EVALUATING BEST LSTM MODEL")
print("="*80)

# Find best LSTM from baseline results
lstm_models = {k: v for k, v in baseline_summary['models'].items() if 'lstm' in k.lower()}
if lstm_models:
    best_lstm_name = min(lstm_models, key=lambda k: lstm_models[k]['overall_smape'])
    print(f"Best LSTM: {best_lstm_name}")
    print(f"Validation sMAPE: {lstm_models[best_lstm_name]['overall_smape']:.2f}%")
    
    # Determine checkpoint path
    if best_lstm_name == 'lstm_v1':
        checkpoint_dir = os.path.join(MODELS_DIR, 'lstm_baseline')
    elif best_lstm_name == 'lstm_v2':
        checkpoint_dir = os.path.join(MODELS_DIR, 'lstm_baseline_v2')
    else:
        checkpoint_dir = None
    
    # Find the checkpoint file
    if checkpoint_dir and os.path.exists(checkpoint_dir):
        ckpt_files = glob.glob(os.path.join(checkpoint_dir, '*.ckpt'))
        
        if ckpt_files:
            lstm_checkpoint = ckpt_files[0]
            print(f"Loading checkpoint: {lstm_checkpoint}")
            
            try:
                # Load checkpoint to get hyperparameters
                checkpoint = torch.load(lstm_checkpoint, map_location='cpu', weights_only=False)
                hparams = checkpoint['hyper_parameters']
                
                print(f"Model hyperparameters:")
                print(f"  Hidden dim: {hparams['hidden_dim']}")
                print(f"  Num layers: {hparams['num_layers']}")
                print(f"  Dropout: {hparams['dropout']}")
                
                # Create a minimal model wrapper for inference
                from darts.models.forecasting.rnn_model import _RNNModule
                
                # Build the model with correct parameters
                model_kwargs = {
                    'name': 'LSTM',
                    'input_size': hparams['input_size'],
                    'hidden_dim': hparams['hidden_dim'],
                    'num_layers': hparams['num_layers'],
                    'target_size': hparams['target_size'],
                    'nr_params': hparams['nr_params'],
                    'dropout': hparams['dropout'],
                    'train_sample_shape': hparams['train_sample_shape']
                }
                
                # Only add optional parameters if they exist
                if 'num_layers_out_fc' in hparams and hparams['num_layers_out_fc']:
                    model_kwargs['num_layers_out_fc'] = hparams['num_layers_out_fc']
                
                model = _RNNModule(**model_kwargs)
                
                # Load weights
                model.load_state_dict(checkpoint['state_dict'])
                model.eval()
                model.to(device)
                
                print(f"✓ Successfully loaded LSTM model!")
                
                # Generate predictions using the dataloader
                all_lstm_forecasts = []
                
                print("Generating LSTM forecasts...")
                with torch.no_grad():
                    for batch in tqdm(test_dataloader, desc="LSTM forecasting"):
                        past, past_mask, future, future_mask, asset_ids, past_ts_list = batch
                        
                        # Move past data to device
                        past = past.to(device)
                        
                        # The model expects input of shape (batch_size, input_chunk_length, input_size)
                        # Check if we need to pad
                        if past.shape[-1] < hparams['input_size']:
                            # Need to pad with zeros for missing covariates
                            batch_size, seq_len, n_features = past.shape
                            n_missing = hparams['input_size'] - n_features
                            padding = torch.zeros(batch_size, seq_len, n_missing, device=device)
                            past_padded = torch.cat([past, padding], dim=-1)
                        else:
                            past_padded = past
                        
                        # Make prediction
                        forecast = model(past_padded)
                        
                        all_lstm_forecasts.append(forecast.cpu())
                
                # Concatenate all predictions
                lstm_forecasts = torch.cat(all_lstm_forecasts, dim=0).to(device)
                
                print(f"\nGenerated {lstm_forecasts.shape[0]} LSTM forecasts")
                
                # Compute sMAPE (without masking)
                lstm_smape = simple_smape(naive_ground_truths_t, lstm_forecasts).item() * 100
                
                # Per-target sMAPE
                lstm_by_target = {}
                for i, target_name in enumerate(TARGET_COLS):
                    target_forecast = lstm_forecasts[:, :, i:i+1]
                    target_gt = naive_ground_truths_t[:, :, i:i+1]
                    target_smape = simple_smape(target_gt, target_forecast).item() * 100
                    lstm_by_target[target_name] = target_smape
                
                print(f"\nLSTM Model sMAPE: {lstm_smape:.2f}%")
                print("\nPer-target sMAPE:")
                for target_name, smape in lstm_by_target.items():
                    print(f"  {target_name:8s}: {smape:.2f}%")
            
            except Exception as e:
                print(f"✗ Error loading/evaluating LSTM model: {e}")
                import traceback
                traceback.print_exc()
                lstm_smape = float('inf')
                lstm_by_target = {}
        else:
            print(f"✗ No checkpoint files found in: {checkpoint_dir}")
            lstm_smape = float('inf')
            lstm_by_target = {}
    else:
        print(f"✗ LSTM checkpoint directory not found: {checkpoint_dir}")
        lstm_smape = float('inf')
        lstm_by_target = {}
else:
    print("No LSTM models found in baseline results")
    lstm_smape = float('inf')
    lstm_by_target = {}

print("="*80)

## 6. Evaluate Chronos2 Models

In [ ]:
print("="*80)
print("EVALUATING CHRONOS2 MODELS")
print("="*80)

# Find best Chronos2 approach
chronos2_approaches = chronos2_summary['approaches']
best_chronos2_approach = min(chronos2_approaches, key=lambda k: chronos2_approaches[k]['overall_smape'])
print(f"Best Chronos2 approach (validation): {best_chronos2_approach}")
print(f"Description: {chronos2_approaches[best_chronos2_approach]['description']}")
print(f"Validation sMAPE: {chronos2_approaches[best_chronos2_approach]['overall_smape']:.2f}%")

# Prepare Chronos2 data from the dataloader
# For Chronos2, we need to organize data by target variable (univariate approach)
print("\nPreparing data for Chronos2 (univariate per target)...")

# Collect all data from dataloader and reorganize by target
chronos2_test_data = {target: {'contexts': [], 'ground_truths': [], 'masks': []} for target in TARGET_COLS}

for batch in tqdm(test_dataloader, desc="Organizing data for Chronos2"):
    past, past_mask, future, future_mask, asset_ids, past_ts_list = batch
    
    # For each target, extract its univariate series
    for target_idx, target_name in enumerate(TARGET_COLS):
        # Extract this target's context and future
        target_context = past[:, :, target_idx].cpu().numpy()  # [batch_size, input_chunk_length]
        target_future = future[:, :, target_idx].cpu().numpy()  # [batch_size, output_chunk_length]
        target_mask = future_mask[:, :, target_idx].cpu().numpy()  # [batch_size, output_chunk_length]
        
        chronos2_test_data[target_name]['contexts'].append(target_context)
        chronos2_test_data[target_name]['ground_truths'].append(target_future)
        chronos2_test_data[target_name]['masks'].append(target_mask)

# Concatenate batches
for target_name in TARGET_COLS:
    chronos2_test_data[target_name]['contexts'] = np.concatenate(chronos2_test_data[target_name]['contexts'], axis=0)
    chronos2_test_data[target_name]['ground_truths'] = np.concatenate(chronos2_test_data[target_name]['ground_truths'], axis=0)
    chronos2_test_data[target_name]['masks'] = np.concatenate(chronos2_test_data[target_name]['masks'], axis=0)
    
    print(f"  {target_name}: {chronos2_test_data[target_name]['contexts'].shape[0]} samples")

### 6.1 Evaluate Zero-Shot Chronos2

In [ ]:
print("\n" + "-"*80)
print("CHRONOS2 ZERO-SHOT UNIVARIATE")
print("-"*80)

# Load base Chronos2 pipeline
chronos2_pipeline = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2",
    device_map="cuda" if torch.cuda.is_available() else None
)
print("✓ Loaded Chronos2 zero-shot pipeline")

# Evaluate each target
chronos2_zeroshot_forecasts = {target: [] for target in TARGET_COLS}

QUANTILE_LEVELS = [0.05, 0.25, 0.5, 0.75, 0.95]
median_idx = len(QUANTILE_LEVELS) // 2

chronos2_pipeline.model.eval()
with torch.no_grad():
    for target_name in TARGET_COLS:
        print(f"\nEvaluating {target_name}...")
        contexts = chronos2_test_data[target_name]['contexts']
        n_samples = contexts.shape[0]
        
        # Predict in batches for efficiency
        batch_size = 32
        for i in tqdm(range(0, n_samples, batch_size), desc=f"Predicting {target_name}"):
            batch_contexts = contexts[i:i+batch_size]
            
            # Convert each context to the format Chronos expects
            batch_inputs = [{'target': ctx} for ctx in batch_contexts]
            
            try:
                q_out, mean_out = chronos2_pipeline.predict_quantiles(
                    batch_inputs,
                    prediction_length=OUTPUT_CHUNK_LENGTH,
                    quantile_levels=QUANTILE_LEVELS
                )
                
                # Extract median forecasts
                for pred in q_out:
                    output_shape = pred.shape
                    if len(output_shape) == 3:
                        forecast = pred[0, :, median_idx]
                    elif len(output_shape) == 2:
                        if output_shape[0] == OUTPUT_CHUNK_LENGTH:
                            forecast = pred[:, median_idx]
                        else:
                            forecast = pred[median_idx, :]
                    else:
                        forecast = pred
                    
                    chronos2_zeroshot_forecasts[target_name].append(forecast)
            
            except Exception as e:
                # If prediction fails, use zeros
                for _ in range(len(batch_contexts)):
                    chronos2_zeroshot_forecasts[target_name].append(np.zeros(OUTPUT_CHUNK_LENGTH))
                continue

# Compute sMAPE per target (without masking)
chronos2_zeroshot_by_target = {}
all_forecasts_zs = []
all_gts_zs = []

for target_name in TARGET_COLS:
    if chronos2_zeroshot_forecasts[target_name]:
        forecasts = np.stack(chronos2_zeroshot_forecasts[target_name], axis=0)
        gts = chronos2_test_data[target_name]['ground_truths']
        
        # Ensure shapes match
        min_len = min(len(forecasts), len(gts))
        forecasts = forecasts[:min_len]
        gts = gts[:min_len]
        
        forecast_t = torch.from_numpy(forecasts).float().to(device).unsqueeze(-1)  # [n, output_len, 1]
        gt_t = torch.from_numpy(gts).float().to(device).unsqueeze(-1)
        
        target_smape = simple_smape(gt_t, forecast_t).item() * 100
        chronos2_zeroshot_by_target[target_name] = target_smape
        
        all_forecasts_zs.append(forecast_t)
        all_gts_zs.append(gt_t)

# Overall sMAPE
if all_forecasts_zs:
    combined_forecasts_zs = torch.cat(all_forecasts_zs, dim=-1)
    combined_gts_zs = torch.cat(all_gts_zs, dim=-1)
    
    chronos2_zeroshot_smape = simple_smape(combined_gts_zs, combined_forecasts_zs).item() * 100
else:
    chronos2_zeroshot_smape = float('inf')

print(f"\nChronos2 Zero-Shot sMAPE: {chronos2_zeroshot_smape:.2f}%")
print("\nPer-target sMAPE:")
for target_name, smape in chronos2_zeroshot_by_target.items():
    print(f"  {target_name:8s}: {smape:.2f}%")

### 6.2 Evaluate Fine-Tuned Chronos2

In [ ]:
print("\n" + "-"*80)
print("CHRONOS2 FINE-TUNED UNIVARIATE")
print("-"*80)

# Load fine-tuned models
finetuned_pipelines = {}
for target_name in TARGET_COLS:
    model_path = os.path.join(MODELS_DIR, f"chronos2_finetuned_{target_name}")
    if os.path.exists(model_path):
        try:
            pipeline = Chronos2Pipeline.from_pretrained(
                model_path,
                device_map="cuda" if torch.cuda.is_available() else None
            )
            finetuned_pipelines[target_name] = pipeline
            print(f"✓ Loaded fine-tuned model for {target_name}")
        except Exception as e:
            print(f"✗ Failed to load fine-tuned model for {target_name}: {e}")
    else:
        print(f"✗ Fine-tuned model not found for {target_name}")

if len(finetuned_pipelines) == 4:
    # Evaluate fine-tuned models
    chronos2_finetuned_forecasts = {target: [] for target in TARGET_COLS}
    
    for target_name in TARGET_COLS:
        print(f"\nEvaluating fine-tuned {target_name}...")
        pipeline = finetuned_pipelines[target_name]
        contexts = chronos2_test_data[target_name]['contexts']
        n_samples = contexts.shape[0]
        
        pipeline.model.eval()
        with torch.no_grad():
            # Predict in batches
            batch_size = 32
            for i in tqdm(range(0, n_samples, batch_size), desc=f"Predicting {target_name}"):
                batch_contexts = contexts[i:i+batch_size]
                
                # Convert to Chronos format
                batch_inputs = [{'target': ctx} for ctx in batch_contexts]
                
                try:
                    q_out, mean_out = pipeline.predict_quantiles(
                        batch_inputs,
                        prediction_length=OUTPUT_CHUNK_LENGTH,
                        quantile_levels=QUANTILE_LEVELS
                    )
                    
                    # Extract median forecasts
                    for pred in q_out:
                        output_shape = pred.shape
                        if len(output_shape) == 3:
                            forecast = pred[0, :, median_idx]
                        elif len(output_shape) == 2:
                            if output_shape[0] == OUTPUT_CHUNK_LENGTH:
                                forecast = pred[:, median_idx]
                            else:
                                forecast = pred[median_idx, :]
                        else:
                            forecast = pred
                        
                        chronos2_finetuned_forecasts[target_name].append(forecast)
                
                except Exception as e:
                    # If prediction fails, use zeros
                    for _ in range(len(batch_contexts)):
                        chronos2_finetuned_forecasts[target_name].append(np.zeros(OUTPUT_CHUNK_LENGTH))
                    continue
    
    # Compute sMAPE (without masking)
    chronos2_finetuned_by_target = {}
    all_forecasts_ft = []
    all_gts_ft = []
    
    for target_name in TARGET_COLS:
        if chronos2_finetuned_forecasts[target_name]:
            forecasts = np.stack(chronos2_finetuned_forecasts[target_name], axis=0)
            gts = chronos2_test_data[target_name]['ground_truths']
            
            # Ensure shapes match
            min_len = min(len(forecasts), len(gts))
            forecasts = forecasts[:min_len]
            gts = gts[:min_len]
            
            forecast_t = torch.from_numpy(forecasts).float().to(device).unsqueeze(-1)
            gt_t = torch.from_numpy(gts).float().to(device).unsqueeze(-1)
            
            target_smape = simple_smape(gt_t, forecast_t).item() * 100
            chronos2_finetuned_by_target[target_name] = target_smape
            
            all_forecasts_ft.append(forecast_t)
            all_gts_ft.append(gt_t)
    
    # Overall sMAPE
    if all_forecasts_ft:
        combined_forecasts_ft = torch.cat(all_forecasts_ft, dim=-1)
        combined_gts_ft = torch.cat(all_gts_ft, dim=-1)
        
        chronos2_finetuned_smape = simple_smape(combined_gts_ft, combined_forecasts_ft).item() * 100
    else:
        chronos2_finetuned_smape = float('inf')
    
    print(f"\nChronos2 Fine-Tuned sMAPE: {chronos2_finetuned_smape:.2f}%")
    print("\nPer-target sMAPE:")
    for target_name, smape in chronos2_finetuned_by_target.items():
        print(f"  {target_name:8s}: {smape:.2f}%")
else:
    print("\n✗ Could not load all fine-tuned models")
    chronos2_finetuned_smape = float('inf')
    chronos2_finetuned_by_target = {}

print("="*80)

## 7. Final Comparison and Results

In [ ]:
print("="*80)
print("FINAL TEST SET RESULTS")
print("="*80)

# Compile all results
final_results = {
    'AutoGluon (Best)': autogluon_smape,
    'Naive Baseline': naive_smape,
    'LSTM (Best)': lstm_smape,
    'Chronos2 Zero-Shot': chronos2_zeroshot_smape,
    'Chronos2 Fine-Tuned': chronos2_finetuned_smape
}

# Sort by performance
sorted_results = sorted(final_results.items(), key=lambda x: x[1])

print("\nOverall sMAPE (ranked):")
for rank, (model, smape) in enumerate(sorted_results, 1):
    if smape != float('inf'):
        print(f"  {rank}. {model:25s}: {smape:.2f}%")
    else:
        print(f"  {rank}. {model:25s}: N/A")

# Best model
best_model_name, best_smape = sorted_results[0]

print(f"\n{'='*80}")
print(f"BEST MODEL: {best_model_name}")
print(f"Test Set sMAPE: {best_smape:.2f}%")
print(f"{'='*80}")

# Per-target comparison
print("\n" + "="*80)
print("PER-TARGET COMPARISON")
print("="*80)

comparison_df = pd.DataFrame({
    'AutoGluon': autogluon_by_target if autogluon_by_target else {t: float('inf') for t in TARGET_COLS},
    'Naive': naive_by_target,
    'LSTM': lstm_by_target if lstm_by_target else {t: float('inf') for t in TARGET_COLS},
    'Chronos2 ZS': chronos2_zeroshot_by_target,
    'Chronos2 FT': chronos2_finetuned_by_target if chronos2_finetuned_by_target else {t: float('inf') for t in TARGET_COLS}
})

print("\n", comparison_df)

# Save results
test_results = {
    'dataset': 'test',
    'num_samples': naive_forecasts.shape[0],  # Use naive_forecasts shape since we have it
    'models': {
        'autogluon_best': {
            'overall_smape': float(autogluon_smape) if autogluon_smape != float('inf') else None,
            'per_target': {k: float(v) for k, v in autogluon_by_target.items()} if autogluon_by_target else None
        },
        'naive_baseline': {
            'overall_smape': float(naive_smape),
            'per_target': {k: float(v) for k, v in naive_by_target.items()}
        },
        'lstm_best': {
            'overall_smape': float(lstm_smape) if lstm_smape != float('inf') else None,
            'per_target': {k: float(v) for k, v in lstm_by_target.items()} if lstm_by_target else None
        },
        'chronos2_zeroshot': {
            'overall_smape': float(chronos2_zeroshot_smape),
            'per_target': {k: float(v) for k, v in chronos2_zeroshot_by_target.items()}
        },
        'chronos2_finetuned': {
            'overall_smape': float(chronos2_finetuned_smape) if chronos2_finetuned_smape != float('inf') else None,
            'per_target': {k: float(v) for k, v in chronos2_finetuned_by_target.items()} if chronos2_finetuned_by_target else None
        }
    },
    'best_model': {
        'name': best_model_name,
        'smape': float(best_smape)
    }
}

results_file = os.path.join(RESULTS_DIR, 'final_test_results.json')
with open(results_file, 'w') as f:
    json.dump(test_results, f, indent=4)

print(f"\n✓ Results saved to: {results_file}")